# Vicena Compute and Rowan workflow bridge

Built with [Vicena](https://vicena.ai). This notebook prepares and inspects remote-compute inputs. It does not submit paid jobs. Vicena Compute handles managed engineering simulation, while Rowan handles molecular modeling.

In [1]:
from pathlib import Path
import sys, json, importlib.metadata, os
root=Path.cwd().resolve()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root/'src'))
from battery_twin import battery_ecm_ngspice_netlist, vicena_compute_sidecar, rowan_electrolyte_request
print('Repository:',root)
print('Vicena Compute SDK:',__import__('vicena_compute').__version__)
print('Rowan SDK:',importlib.metadata.version('rowan-python'))
print('Gateway configured:',bool(os.getenv('VICENA_EXTERNAL_API_KEY')))

Repository: /root/sessions/481_repository-analysis-inquiry/battery-degradation-state-of-health-digital-twin
Vicena Compute SDK: 0.6.1
Rowan SDK: 3.1.8
Gateway configured: True


In [2]:
netlist=battery_ecm_ngspice_netlist()
sidecar=vicena_compute_sidecar()
print(netlist)
print('Limits:',sidecar[0]['limits'])
print('Scientific boundary: illustrative battery ECM parameters, managed ngspice execution requires explicit submission.')

Battery 1RC Thevenin pulse response, illustrative parameters
V_OCV ocv 0 3.70
I_LOAD out 0 PULSE(0 3.0 1m 1u 1u 90.0 180.0)
R0 ocv n1 0.032
R1 n1 out 0.018
C1 n1 out 2400.0
.tran 0.3 180.0
.print tran time v(out) v(n1)
.measure tran v_terminal_mid FIND v(out) AT=60.0
.measure tran v_terminal_min MIN v(out)
.end

Limits: {'cpu_cores': 1, 'memory_mib': 1024, 'wall_seconds': 120}
Scientific boundary: illustrative battery ECM parameters, managed ngspice execution requires explicit submission.


In [3]:
rowan_request=rowan_electrolyte_request()
print(json.dumps(rowan_request,indent=2))
print('Molecule: ethylene carbonate, SMILES O=C1OCO1')
print('Scientific boundary: descriptor screening does not establish electrolyte stability, conductivity, electrochemical window, or cell performance.')

{
  "provider": "rowan",
  "workflow": "descriptors",
  "initial_smiles": "O=C1OCO1",
  "name": "battery-electrolyte-screen",
  "max_vicena_credits": 100,
  "submission": "requires explicit user authorization",
  "task_key": "7b2e073b041558142ad3108f70cd965b673ab0df36a04611e75b13e935a794f3"
}
Molecule: ethylene carbonate, SMILES O=C1OCO1
Scientific boundary: descriptor screening does not establish electrolyte stability, conductivity, electrochemical window, or cell performance.


## Safe execution route

1. Review the generated ECM netlist and molecule identity.
2. Set an explicit computation-credit budget.
3. Use Vicena's trusted managed runner for the ngspice workflow.
4. Use Vicena's Rowan runner for molecular workflows, preserving `rowan_workflows.json`.
5. Save provider job IDs, results, logs, units, and limitations. Never put paid submission calls in notebook cells.